In [ ]:
import os
import torch
import shutil
import numpy as np
from torch import nn
from datasets import load_dataset
from transformers import (
    BertPreTrainedModel, BertModel, BertTokenizerFast,
    Trainer, TrainingArguments, TrainerCallback, logging,
    DataCollatorForTokenClassification
)
from sklearn.metrics import f1_score

# 仅显示错误信息，保持控制台整洁
logging.set_verbosity_error()


# ===================== 1. 模型定义 (BERT + BiLSTM) =====================
class BertBiLSTMForNER(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(0.3)

        self.bilstm = nn.LSTM(
            input_size=config.hidden_size,
            hidden_size=config.hidden_size // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        sequence_output = self.dropout(outputs.last_hidden_state)
        lstm_output, _ = self.bilstm(sequence_output)
        logits = self.classifier(lstm_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits}


# ===================== 2. 数据处理与评估 =====================
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=128)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = [
            label[word_idx] if word_idx is not None and word_idx != (word_ids[j - 1] if j > 0 else -1) else -100
            for j, word_idx in enumerate(word_ids)]
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def compute_metrics(eval_preds, label_list):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in
                        zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in
                   zip(predictions, labels)]
    return {"f1": f1_score([item for sublist in true_labels for item in sublist],
                           [item for sublist in true_predictions for item in sublist], average="weighted")}


# ===================== 3. 自定义进度回调 =====================
class ProgressPrinterCallback(TrainerCallback):
    """一个简单的回调，用于在日志步数打印更友好的信息"""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"🔹 Step: {state.global_step:5d} | Train Loss: {logs['loss']:.4f}")
            if "eval_loss" in logs:
                print(
                    f"✅ Epoch {logs.get('epoch', 0):.1f} Eval -> Loss: {logs['eval_loss']:.4f} | F1: {logs.get('eval_f1', 0):.4f}")


# ===================== 4. 主程序 =====================
def main():
    MODEL_PATH = "bert-base-chinese"
    OUTPUT_DIR = "./results"

    print("📦 正在加载数据并初始化分词器...")
    dataset = load_dataset("PassbyGrocer/msra-ner")
    label_list = dataset["train"].features["ner_tags"].feature.names
    tokenizer = BertTokenizerFast.from_pretrained(MODEL_PATH)

    tokenized_ds = dataset.map(lambda x: tokenize_and_align_labels(x, tokenizer), batched=True,
                               remove_columns=dataset["train"].column_names)
    model = BertBiLSTMForNER.from_pretrained(MODEL_PATH, num_labels=len(label_list))

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=32,
        learning_rate=3e-5,
        logging_steps=20,  # 每 20 步打印一次 Loss，让你看到实时的变化
        eval_strategy="epoch",  # 每轮结束后进行评估
        save_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="tensorboard",  # 开启 TensorBoard 支持
        logging_dir="./logs",  # 日志保存路径
        disable_tqdm=False,  # 确保进度条开启
        load_best_model_at_end=True,
        metric_for_best_model="f1"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=lambda x: compute_metrics(x, label_list),
        callbacks=[ProgressPrinterCallback()]  # 添加自定义打印回调
    )

    print("🚀 开始训练...")
    trainer.train()

    # 保存结果
    trainer.save_model("final_model")
    print("✅ 训练完成，模型已保存！")


if __name__ == "__main__":
    main()